#Comparación de modelos de clasificación para datos de contaminante aereo NOx y datos de urgencias respiratorios en la quinta región

## 1. Imports

In [0]:
from functools import reduce
import re
import unicodedata

from pyspark.sql import functions as F
from pyspark.sql import Window

## 2. Carga de datos

In [0]:
CATALOG = "upla"
SCHEMA = "mcdma_analisis_datos_ma"
VOLUME = "aire_salud_volume"

BASE_PATH = f"/Volumes/{CATALOG}/{SCHEMA}/{VOLUME}"
SINCA_DIR = f"{BASE_PATH}/raw/sinca"
URGENCIAS_DIR = f"{BASE_PATH}/raw/urgencias"

BRONZE_SINCA_TABLE = f"{CATALOG}.{SCHEMA}.bronze_sinca_nox_diario"
BRONZE_URGENCIAS_TABLE = f"{CATALOG}.{SCHEMA}.bronze_urgencias"
SILVER_SINCA_TABLE = f"{CATALOG}.{SCHEMA}.silver_sinca_nox_semanal"
SILVER_URGENCIAS_TABLE = f"{CATALOG}.{SCHEMA}.silver_urgencias_resp_semanal"
GOLD_TABLE = f"{CATALOG}.{SCHEMA}.gold_aire_salud_semanal"

spark.sql(f"USE CATALOG {CATALOG}")
spark.sql(f"USE SCHEMA {SCHEMA}")

print("BASE_PATH:", BASE_PATH)
print("SINCA_DIR:", SINCA_DIR)
print("URGENCIAS_DIR:", URGENCIAS_DIR)

In [0]:
display(dbutils.fs.ls(SINCA_DIR))
display(dbutils.fs.ls(URGENCIAS_DIR))

In [0]:
def clean_col_name(c):
    c = c.strip()
    c = unicodedata.normalize("NFKD", c).encode("ascii", "ignore").decode("ascii")
    c = re.sub(r"[^0-9A-Za-z]+", "_", c)
    c = c.strip("_").lower()
    return c


def make_unique_columns(cols):
    clean_cols = []
    seen = {}

    for i, c in enumerate(cols):
        name = clean_col_name(c)

        if not name:
            name = f"empty_col_{i}"

        if name in seen:
            seen[name] += 1
            name = f"{name}_{seen[name]}"
        else:
            seen[name] = 0

        clean_cols.append(name)

    return clean_cols


def read_csv_folder(folder_path, encoding="UTF-8"):
    files = [
        f.path for f in dbutils.fs.ls(folder_path)
        if f.name.lower().endswith(".csv")
    ]

    print("Archivos encontrados:", len(files))

    dfs = []

    for path in files:
        print("Leyendo:", path)

        df = (
            spark.read
            .format("csv")
            .option("header", "true")
            .option("sep", ";")
            .option("encoding", encoding)
            .option("inferSchema", "false")
            .load(path)
        )

        df = df.toDF(*make_unique_columns(df.columns))
        df = df.withColumn("source_file", F.lit(path.split("/")[-1]))
        dfs.append(df)

    all_cols = sorted(set().union(*[set(df.columns) for df in dfs]))

    dfs_aligned = [
        df.select([
            F.col(c) if c in df.columns else F.lit(None).alias(c)
            for c in all_cols
        ])
        for df in dfs
    ]

    return reduce(lambda a, b: a.unionByName(b), dfs_aligned)


def to_double_comma(col_name):
    return (
        F.regexp_replace(
            F.when(F.trim(F.col(col_name).cast("string")) == "", None)
             .otherwise(F.trim(F.col(col_name).cast("string"))),
            ",",
            "."
        )
        .cast("double")
    )


def to_long_safe(col_name):
    return (
        F.regexp_replace(
            F.when(F.trim(F.col(col_name).cast("string")) == "", None)
             .otherwise(F.trim(F.col(col_name).cast("string"))),
            ",",
            "."
        )
        .cast("double")
        .cast("long")
    )

In [0]:
sinca_raw = read_csv_folder(SINCA_DIR, encoding="UTF-8")

print(sinca_raw.columns)
display(sinca_raw.limit(10))

In [0]:
# Esta configuracion es porque los archivos pesan hasta 1.7 GB y se deben cargar de a poco para revisar y despues filtrar
spark.conf.set("spark.sql.shuffle.partitions", "64")
spark.conf.set("spark.sql.files.maxPartitionBytes", 64 * 1024 * 1024)

print("Configuración lista para archivos grandes")

In [0]:
def read_urgencias_file(path, encoding="UTF-8"):
    print("Leyendo:", path)

    df = (
        spark.read
        .format("csv")
        .option("header", "true")
        .option("sep", ";")
        .option("encoding", encoding)
        .option("inferSchema", "false")
        .load(path)
    )

    df = df.toDF(*make_unique_columns(df.columns))

    required_cols = [
        "idestablecimiento",
        "nestablecimiento",
        "idcausa",
        "glosacausa",
        "total",
        "menores_1",
        "de_1_a_4",
        "de_5_a_14",
        "de_15_a_64",
        "de_65_y_mas",
        "fecha",
        "semana",
        "glosatipoestablecimiento",
        "glosatipoatencion",
        "glosatipocampana",
        "codigoregion",
        "nombreregion",
        "codigodependencia",
        "nombredependencia",
        "codigocomuna",
        "nombrecomuna"
    ]

    df = df.select([
        F.col(c) if c in df.columns else F.lit(None).alias(c)
        for c in required_cols
    ])

    df = (
        df
        .withColumn("source_file", F.lit(path.split("/")[-1]))
        .withColumn("fecha", F.to_date(F.col("fecha"), "dd/MM/yyyy"))
        .withColumn("anio", F.year("fecha"))
        .withColumn("semana_minsal", F.col("semana").cast("int"))
        .withColumn("idcausa_int", F.col("idcausa").cast("int"))
        .withColumn("codigoregion_int", F.col("codigoregion").cast("int"))
        .withColumn("codigocomuna_int", F.col("codigocomuna").cast("int"))
        .withColumn("total_num", to_long_safe("total"))
        .withColumn("menores_1_num", to_long_safe("menores_1"))
        .withColumn("de_1_a_4_num", to_long_safe("de_1_a_4"))
        .withColumn("de_5_a_14_num", to_long_safe("de_5_a_14"))
        .withColumn("de_15_a_64_num", to_long_safe("de_15_a_64"))
        .withColumn("de_65_y_mas_num", to_long_safe("de_65_y_mas"))
        .filter(F.col("fecha").isNotNull())
        .filter((F.col("anio") >= 2020) & (F.col("anio") <= 2025))
        .filter(
            (F.col("codigoregion_int") == 5) |
            (F.upper(F.col("nombreregion")).contains("VALPAR"))
        )
    )

    return df

def select_first_existing(df, candidates, alias_name):
    """
    Selecciona la primera columna existente entre varios nombres posibles.
    Si ninguna existe, crea la columna como null.
    """
    for c in candidates:
        if c in df.columns:
            return F.col(c).alias(alias_name)
    return F.lit(None).alias(alias_name)


def read_urgencias_file_v2(path, encoding="ISO-8859-1"):
    print("Leyendo:", path.split("/")[-1])

    df = (
        spark.read
        .format("csv")
        .option("header", "true")
        .option("sep", ";")
        .option("encoding", encoding)
        .option("inferSchema", "false")
        .load(path)
    )

    df = df.toDF(*make_unique_columns(df.columns))

    df = df.select(
        select_first_existing(df, ["idestablecimiento", "id_establecimiento"], "idestablecimiento"),
        select_first_existing(df, ["nestablecimiento", "n_establecimiento", "nombre_establecimiento"], "nestablecimiento"),
        select_first_existing(df, ["idcausa", "id_causa"], "idcausa"),
        select_first_existing(df, ["glosacausa", "glosa_causa"], "glosacausa"),
        select_first_existing(df, ["total"], "total"),
        select_first_existing(df, ["menores_1", "menor_a_1", "menor_1"], "menores_1"),
        select_first_existing(df, ["de_1_a_4", "column7"], "de_1_a_4"),
        select_first_existing(df, ["de_5_a_14", "_14"], "de_5_a_14"),
        select_first_existing(df, ["de_15_a_64", "_5_64"], "de_15_a_64"),
        select_first_existing(df, ["de_65_y_mas", "_5_mas", "de_65_y_mas_"], "de_65_y_mas"),
        select_first_existing(df, ["fecha"], "fecha"),
        select_first_existing(df, ["semana"], "semana"),
        select_first_existing(df, ["glosatipoestablecimiento"], "glosatipoestablecimiento"),
        select_first_existing(df, ["glosatipoatencion"], "glosatipoatencion"),
        select_first_existing(df, ["glosatipocampana"], "glosatipocampana"),
        select_first_existing(df, ["codigoregion", "codigo_region"], "codigoregion"),
        select_first_existing(df, ["nombreregion", "nombre_region"], "nombreregion"),
        select_first_existing(df, ["codigodependencia", "codigo_dependencia"], "codigodependencia"),
        select_first_existing(df, ["nombredependencia", "nombre_dependencia"], "nombredependencia"),
        select_first_existing(df, ["codigocomuna", "codigo_comuna"], "codigocomuna"),
        select_first_existing(df, ["nombrecomuna", "nombre_comuna"], "nombrecomuna")
    )

    df = (
        df
        .withColumn("source_file", F.lit(path.split("/")[-1]))
        .withColumn("fecha", F.to_date(F.col("fecha"), "dd/MM/yyyy"))
        .withColumn("anio", F.year("fecha"))
        .withColumn("semana_minsal", F.col("semana").cast("int"))
        .withColumn("idcausa_int", F.col("idcausa").cast("int"))
        .withColumn("codigoregion_int", F.col("codigoregion").cast("int"))
        .withColumn("codigocomuna_int", F.col("codigocomuna").cast("int"))
        .withColumn("total_num", to_long_safe("total"))
        .withColumn("menores_1_num", to_long_safe("menores_1"))
        .withColumn("de_1_a_4_num", to_long_safe("de_1_a_4"))
        .withColumn("de_5_a_14_num", to_long_safe("de_5_a_14"))
        .withColumn("de_15_a_64_num", to_long_safe("de_15_a_64"))
        .withColumn("de_65_y_mas_num", to_long_safe("de_65_y_mas"))
        .filter(F.col("fecha").isNotNull())
        .filter((F.col("anio") >= 2020) & (F.col("anio") <= 2025))
    )

    return df

In [0]:
urg_files = [
    f.path for f in dbutils.fs.ls(URGENCIAS_DIR)
    if f.name.lower().endswith(".csv")
]

print("Archivos de urgencias:", len(urg_files))
for f in urg_files:
    print(f)

urg_dfs = [read_urgencias_file(path, encoding="UTF-8") for path in urg_files]

urg_valpo = reduce(lambda a, b: a.unionByName(b), urg_dfs)

print("DataFrame urg_valpo creado")

## 3. Preprocesamiento

In [0]:
sinca_clean = (
    sinca_raw
    .withColumn(
        "fecha_str",
        F.lpad(F.col("fecha_yymmdd").cast("string"), 6, "0")
    )
    .withColumn(
        "fecha",
        F.to_date(
            F.concat(
                F.lit("20"),
                F.substring("fecha_str", 1, 2),
                F.substring("fecha_str", 3, 2),
                F.substring("fecha_str", 5, 2)
            ),
            "yyyyMMdd"
        )
    )
    .withColumn(
        "estacion",
        F.regexp_replace(
            F.regexp_replace(F.col("source_file"), r"^v_nox_diario_", ""),
            r"_2018_2025\.csv$",
            ""
        )
    )
    .withColumn("valor_validado", to_double_comma("registros_validados"))
    .withColumn("valor_preliminar", to_double_comma("registros_preliminares"))
    .withColumn("valor_no_validado", to_double_comma("registros_no_validados"))
    .withColumn(
        "valor_nox",
        F.coalesce("valor_validado", "valor_preliminar", "valor_no_validado")
    )
    .withColumn(
        "tipo_registro",
        F.when(F.col("valor_validado").isNotNull(), F.lit("validado"))
         .when(F.col("valor_preliminar").isNotNull(), F.lit("preliminar"))
         .when(F.col("valor_no_validado").isNotNull(), F.lit("no_validado"))
         .otherwise(F.lit("sin_dato"))
    )
    .filter(F.col("fecha").isNotNull())
    .filter(F.col("valor_nox").isNotNull())
    .filter((F.col("fecha") >= F.lit("2020-01-01")) & (F.col("fecha") <= F.lit("2025-12-31")))
    .select(
        "fecha",
        "estacion",
        F.lit("NOx").alias("contaminante"),
        "valor_nox",
        "tipo_registro",
        "source_file"
    )
)

display(sinca_clean.limit(20))

In [0]:
(
    sinca_clean
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(BRONZE_SINCA_TABLE)
)

print("Tabla creada:", BRONZE_SINCA_TABLE)

In [0]:
(
    urg_valpo
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(BRONZE_URGENCIAS_TABLE)
)

print("Tabla creada:", BRONZE_URGENCIAS_TABLE)

In [0]:
urg_valpo = spark.table(BRONZE_URGENCIAS_TABLE)

In [0]:
urg_resp = urg_valpo.filter(F.col("idcausa_int") == 2)

In [0]:
# Los años 2020 a 2022 no traen el campo comuna, por lo que hay que obtenerlo mediante el codigo de comuna

for path in urg_files:
    df_tmp = (
        spark.read
        .format("csv")
        .option("header", "true")
        .option("sep", ";")
        .option("encoding", "ISO-8859-1")
        .option("inferSchema", "false")
        .load(path)
    )

    print("\nArchivo:", path.split("/")[-1])
    print(make_unique_columns(df_tmp.columns))

In [0]:
urg_dfs_v2 = [read_urgencias_file_v2(path, encoding="ISO-8859-1") for path in urg_files]

urg_all = reduce(lambda a, b: a.unionByName(b), urg_dfs_v2)

print("DataFrame urg_all creado")

In [0]:
# Se construye una tabla puente para unificar los datos que tienen la comuna como texto y otros como numero

establecimientos_base = (
    urg_all
    .filter(F.col("codigoregion_int").isNotNull())
    .filter(F.col("codigocomuna_int").isNotNull())
    .withColumn("idestablecimiento_norm", F.trim(F.col("idestablecimiento")))
    .groupBy(
        "idestablecimiento_norm",
        "codigoregion_int",
        "nombreregion",
        "codigocomuna_int",
        "nombrecomuna"
    )
    .agg(F.count("*").alias("n_apariciones"))
)

w_est = Window.partitionBy("idestablecimiento_norm").orderBy(F.desc("n_apariciones"))

establecimientos_map = (
    establecimientos_base
    .withColumn("rn", F.row_number().over(w_est))
    .filter(F.col("rn") == 1)
    .select(
        F.col("idestablecimiento_norm"),
        F.col("codigoregion_int").alias("map_codigoregion"),
        F.col("nombreregion").alias("map_nombreregion"),
        F.col("codigocomuna_int").alias("map_codigocomuna"),
        F.col("nombrecomuna").alias("map_nombrecomuna")
    )
)

display(establecimientos_map.limit(20))

In [0]:
urg_all_norm = (
    urg_all
    .withColumn("idestablecimiento_norm", F.trim(F.col("idestablecimiento")))
)

urg_all_filled = (
    urg_all_norm.alias("u")
    .join(
        establecimientos_map.alias("m"),
        on="idestablecimiento_norm",
        how="left"
    )
    .withColumn(
        "codigoregion_final",
        F.coalesce(F.col("u.codigoregion_int"), F.col("m.map_codigoregion"))
    )
    .withColumn(
        "nombreregion_final",
        F.coalesce(F.col("u.nombreregion"), F.col("m.map_nombreregion"))
    )
    .withColumn(
        "codigocomuna_final",
        F.coalesce(F.col("u.codigocomuna_int"), F.col("m.map_codigocomuna"))
    )
    .withColumn(
        "nombrecomuna_final",
        F.coalesce(F.col("u.nombrecomuna"), F.col("m.map_nombrecomuna"))
    )
)

In [0]:
(
    urg_valpo
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(BRONZE_URGENCIAS_TABLE)
)

print("Tabla corregida:", BRONZE_URGENCIAS_TABLE)

## 4. Entrenamiento

No hay celdas de entrenamiento en la version actual del notebook.

## 5. Evaluacion

In [0]:
display(
    sinca_clean
    .groupBy("tipo_registro")
    .count()
)

display(
    sinca_clean
    .groupBy("estacion")
    .agg(
        F.count("*").alias("n_registros"),
        F.min("fecha").alias("fecha_min"),
        F.max("fecha").alias("fecha_max"),
        F.avg("valor_nox").alias("nox_promedio"),
        F.max("valor_nox").alias("nox_maximo")
    )
    .orderBy("estacion")
)

In [0]:
display(
    urg_valpo
    .groupBy("anio", "nombreregion")
    .agg(
        F.sum("total_num").alias("total_atenciones"),
        F.count("*").alias("filas")
    )
    .orderBy("anio")
)

In [0]:
resp_candidates = (
    urg_valpo
    .filter(
        F.upper(F.col("glosacausa")).contains("RESPIRATOR") |
        F.upper(F.col("glosacausa")).contains("BRONQUIT") |
        F.upper(F.col("glosacausa")).contains("NEUMON") |
        F.upper(F.col("glosacausa")).contains("INFLUENZA") |
        F.upper(F.col("glosacausa")).contains("IRA")
    )
    .groupBy("anio", "idcausa_int", "glosacausa")
    .agg(
        F.sum("total_num").alias("total"),
        F.count("*").alias("filas")
    )
    .orderBy("anio", "idcausa_int")
)

display(resp_candidates)

In [0]:
display(
    urg_all
    .groupBy("anio", "source_file")
    .agg(
        F.count("*").alias("filas"),
        F.sum(F.when(F.col("codigoregion_int").isNotNull(), 1).otherwise(0)).alias("filas_con_region"),
        F.sum(F.when(F.col("codigocomuna_int").isNotNull(), 1).otherwise(0)).alias("filas_con_comuna")
    )
    .orderBy("anio", "source_file")
)

In [0]:
print("Establecimientos mapeados:", establecimientos_map.count())

display(
    establecimientos_map
    .groupBy("map_codigoregion", "map_nombreregion")
    .count()
    .orderBy("map_codigoregion")
)

In [0]:
display(
    urg_all_filled
    .groupBy("anio")
    .agg(
        F.count("*").alias("filas"),
        F.sum(F.when(F.col("codigoregion_final").isNotNull(), 1).otherwise(0)).alias("filas_con_region_final"),
        F.sum(F.when(F.col("codigocomuna_final").isNotNull(), 1).otherwise(0)).alias("filas_con_comuna_final"),
        F.countDistinct("idestablecimiento_norm").alias("n_establecimientos")
    )
    .orderBy("anio")
)

In [0]:
display(
    urg_valpo
    .groupBy("anio")
    .agg(
        F.countDistinct("idestablecimiento_norm").alias("n_establecimientos_valpo"),
        F.countDistinct("codigocomuna_final").alias("n_comunas_valpo"),
        F.sum("total_num").alias("total_atenciones")
    )
    .orderBy("anio")
)

## 6. Visualizaciones

In [0]:
display(spark.table(BRONZE_SINCA_TABLE).limit(10))